# SBS validation: OMTPL Retail Classical FREQ

Ноутбук запускает полный HTML-отчет SBS для файла `OMTPL_Retail_Classical_FREQ_Inference_db_ver01.csv`.

Для модели частоты в SBS нужно подавать чистую частоту `GLM_FREQ` как `score_column`, а экспозицию `Exposure` отдельной `weight_column`; колонка `GLM_FREQ_weighted = GLM_FREQ * Exposure` оставлена ниже только для контрольной диагностики, потому что тесты сами взвешивают rate на экспозицию там, где это требуется.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_FILENAME = "OMTPL_Retail_Classical_FREQ_Inference_db_ver01.csv"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / DATA_FILENAME
REPORTS_DIR = PROJECT_ROOT / "reports" / "omtpl_retail_classical_freq"

if not DATA_PATH.exists():
    candidates = sorted(PROJECT_ROOT.rglob(DATA_FILENAME))
    if candidates:
        DATA_PATH = candidates[0]

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"REPORTS_DIR: {REPORTS_DIR}")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Не найден {DATA_FILENAME}. Положите файл в data/raw/ или рядом с проектом, "
        "либо поправьте DATA_PATH в этой ячейке."
    )

## Загрузка данных

In [ ]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    attempts = [
        {"sep": None, "engine": "python"},
        {"sep": ";", "encoding": "cp1251", "decimal": ","},
        {"sep": ";", "encoding": "utf-8", "decimal": ","},
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ",", "encoding": "cp1251"},
    ]
    errors = []
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            if df.shape[1] > 1:
                print(f"read_csv kwargs: {kwargs}")
                return df
        except Exception as exc:
            errors.append(f"{kwargs}: {exc}")
    raise RuntimeError("Не удалось прочитать CSV:\n" + "\n".join(errors))

data = read_csv_safely(DATA_PATH)
print(f"Shape: {data.shape[0]:,} rows x {data.shape[1]:,} columns")
display(data.head())
display(pd.DataFrame({"column": data.columns, "dtype": data.dtypes.astype(str).values}).head(120))

## Настройка колонок

Если в файле названия отличаются, обычно достаточно поправить кандидаты ниже.

In [ ]:
def pick_column(df: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    by_lower = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        found = by_lower.get(candidate.lower())
        if found is not None:
            return found
    if required:
        raise KeyError(
            "Не найдена колонка из списка кандидатов: "
            + ", ".join(candidates)
            + "\nПохожие колонки: "
            + ", ".join([c for c in df.columns if any(x.lower() in c.lower() for x in candidates[:3])][:20])
        )
    return None

SCORE_COLUMN = pick_column(data, ["GLM_FREQ", "GLM FREQ", "GLM_DAGO_Freq", "GLM_Freq"])
WEIGHTED_SCORE_COLUMN = pick_column(data, ["GLM_FREQ_weighted", "GLM_MODEL_weighted", "GLM_WEIGHTED_FREQ"], required=False)
WEIGHT_COLUMN = pick_column(data, ["Exposure", "EXPOSURE", "expo", "WEIGHT", "weight"])
SAMPLE_COLUMN = pick_column(data, ["db_slpit", "DB_SLPIT", "db_split", "DB_SPLIT"])
DATE_COLUMN = pick_column(data, ["TIME_TREND_SEAS_REPORT_YEAR_QUARTER", "TIME_TREND_ID_START_QUARTER", "SEAS_REPORT_YEAR_QUARTER", "ID_START_QUARTER"])
TARGET_COLUMN = pick_column(data, ["ClaimsCountPaid", "CLAIMS_COUNT_PAID", "DAGO_ClaimsCount", "ClaimCount", "ClaimsCount", "TARGET", "target"])
ID_COLUMN = pick_column(data, ["UNIQUE_KEY", "ID_AGRID", "POLICY_ID", "POLICY_RK", "ID", "id"], required=False)

if ID_COLUMN is None:
    ID_COLUMN = "__ROW_ID__"
    data[ID_COLUMN] = np.arange(len(data), dtype=np.int64)
    print("WARNING: id_column не найден, создан технический __ROW_ID__. Если есть реальный ключ полиса, лучше указать его явно.")

selected_columns = {
    "score_column": SCORE_COLUMN,
    "weighted_score_diagnostic": WEIGHTED_SCORE_COLUMN,
    "weight_column": WEIGHT_COLUMN,
    "sample_column": SAMPLE_COLUMN,
    "date_column": DATE_COLUMN,
    "target_column": TARGET_COLUMN,
    "id_column": ID_COLUMN,
}
selected_columns

## Подготовка данных под контракт SBS

In [ ]:
def to_numeric_strict(series: pd.Series, column: str) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        out = pd.to_numeric(series, errors="coerce")
    else:
        normalized = (
            series.astype(str)
            .str.strip()
            .str.replace("\u00a0", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        out = pd.to_numeric(normalized, errors="coerce")
    bad = out.isna() & series.notna() & ~series.astype(str).str.strip().isin(["", "nan", "None"])
    if bad.any():
        examples = series[bad].astype(str).head(10).tolist()
        raise ValueError(f"Колонка {column} не приводится к numeric, примеры: {examples}")
    return out

def parse_report_date(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series, errors="raise")
    for fmt in ("%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d"):
        parsed = pd.to_datetime(series, format=fmt, errors="coerce")
        if parsed.notna().mean() >= 0.95:
            return parsed
    return pd.to_datetime(series, errors="raise", dayfirst=True)

final_df = data.copy()

final_df[DATE_COLUMN] = parse_report_date(final_df[DATE_COLUMN])
final_df[SAMPLE_COLUMN] = final_df[SAMPLE_COLUMN].astype(str).str.strip().str.upper()
final_df[ID_COLUMN] = final_df[ID_COLUMN].astype(str)

for column in [TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN]:
    final_df[column] = to_numeric_strict(final_df[column], column)
    if final_df[column].isna().any():
        raise ValueError(f"В обязательной колонке {column} есть пропуски: {int(final_df[column].isna().sum())}")

if (final_df[WEIGHT_COLUMN] <= 0).any():
    bad_n = int((final_df[WEIGHT_COLUMN] <= 0).sum())
    raise ValueError(f"В колонке экспозиции {WEIGHT_COLUMN} есть неположительные значения: {bad_n}")

if WEIGHTED_SCORE_COLUMN is not None:
    final_df[WEIGHTED_SCORE_COLUMN] = to_numeric_strict(final_df[WEIGHTED_SCORE_COLUMN], WEIGHTED_SCORE_COLUMN)
    weighted_expected = final_df[SCORE_COLUMN] * final_df[WEIGHT_COLUMN]
    max_abs_diff = (final_df[WEIGHTED_SCORE_COLUMN] - weighted_expected).abs().max()
    print(f"Weighted prediction check: max |{WEIGHTED_SCORE_COLUMN} - {SCORE_COLUMN} * {WEIGHT_COLUMN}| = {max_abs_diff:.6g}")

split_summary = final_df[SAMPLE_COLUMN].value_counts(dropna=False).rename_axis(SAMPLE_COLUMN).reset_index(name="rows")
date_summary = pd.DataFrame({
    "metric": ["min_date", "max_date", "missing_dates"],
    "value": [final_df[DATE_COLUMN].min(), final_df[DATE_COLUMN].max(), final_df[DATE_COLUMN].isna().sum()],
})

display(split_summary)
display(date_summary)
display(final_df[[TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN]].describe().T)

## Список факторов

По умолчанию в отчет идут `*_SCORING` как числовые факторы; `GLM_RF_levels`, если есть в файле, идет как категориальный фактор.

In [ ]:
service_columns = {
    SCORE_COLUMN,
    WEIGHTED_SCORE_COLUMN,
    WEIGHT_COLUMN,
    SAMPLE_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
    ID_COLUMN,
    "GLM_MODEL_BINQ10",
    "GLM_MODEL_BIN_Q10",
    "GLM_MODEL_weighted_BINQ10",
    "GLM_MODEL_WEIGHTED_BINQ10",
    "GLM_MODEL_weighted_BIN_Q10",
    "INTERCEPT_VALUE",
}
service_columns = {c for c in service_columns if c is not None}

scored_factor_columns = sorted([
    c for c in final_df.columns
    if c.endswith("_SCORING") and c not in service_columns
])
value_factor_columns = sorted([
    c for c in final_df.columns
    if c.endswith("_VALUE") and c not in service_columns and c != "INTERCEPT_VALUE"
])

FEATURE_SET = "scoring"  # "scoring", "value" или "both"
if FEATURE_SET == "scoring":
    numeric_features = scored_factor_columns
elif FEATURE_SET == "value":
    numeric_features = value_factor_columns
elif FEATURE_SET == "both":
    numeric_features = list(dict.fromkeys([*scored_factor_columns, *value_factor_columns]))
else:
    raise ValueError("FEATURE_SET должен быть 'scoring', 'value' или 'both'")

categorical_features = [
    c for c in ["GLM_RF_levels", "GLM_RF_LEVELS"]
    if c in final_df.columns and c not in service_columns
]

if not numeric_features:
    raise ValueError("Не найдены числовые факторы *_SCORING или *_VALUE. Проверьте названия колонок и FEATURE_SET.")

feature_missing = []
for column in numeric_features:
    final_df[column] = to_numeric_strict(final_df[column], column)
    missing = int(final_df[column].isna().sum())
    if missing:
        feature_missing.append({"feature": column, "missing_filled_with_zero": missing})
        final_df[column] = final_df[column].fillna(0.0)

for column in categorical_features:
    final_df[column] = final_df[column].astype(str).fillna("__MISSING__")

print(f"FEATURE_SET: {FEATURE_SET}")
print(f"Scored factors: {len(scored_factor_columns)}")
print(f"Value factors: {len(value_factor_columns)}")
print(f"Numeric features used: {len(numeric_features)}")
print(f"Categorical features used: {len(categorical_features)}")

if feature_missing:
    print(f"WARNING: пропуски в {len(feature_missing)} numeric-факторах заполнены 0.0 для стабильного запуска M2/M3.")
    display(pd.DataFrame(feature_missing).head(80))

display(pd.DataFrame({
    "numeric_features": pd.Series(numeric_features),
    "categorical_features": pd.Series(categorical_features),
}).head(120))

## Конфиг SBS

In [ ]:
config = {
    "model_mode": "frequency",
    "validation_mode": "oos",
    "columns": {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "score_column": SCORE_COLUMN,
        "prediction_column": None,
        "date_column": DATE_COLUMN,
        "target_column": TARGET_COLUMN,
        "sample_column": SAMPLE_COLUMN,
        "id_column": ID_COLUMN,
        "weight_column": WEIGHT_COLUMN,
    },
}

config

In [ ]:
from sbs_evaluation_tool.core.config_validation import validate_config

validated_config = validate_config(config, final_df)
print(validated_config)
print(f"Numeric features: {len(validated_config.columns.numeric_features)}")
print(f"Categorical features: {len(validated_config.columns.categorical_features)}")

## Smoke-check одного теста

Ячейку можно пропустить, если нужно сразу запускать полный отчет.

In [ ]:
from sbs_evaluation_tool.core.inline_display import show_test_inline
from sbs_evaluation_tool.tests.m4_model_performance import M41_TargetRateTest

show_test_inline(M41_TargetRateTest(), {"df": final_df, "config": config})

## Полный запуск M0-M5 и генерация HTML-отчета

`RUN_SIZE = "normal"` запускает все группы, включая M3; если датасет большой и нужен быстрый первый HTML, поставьте `RUN_SIZE = "heavy"`, в текущем runner это пропускает M3.

In [ ]:
from sbs_evaluation_tool.core.runner import run_sbs_evaluation

RUN_SIZE = "normal"  # "normal" = полный M0-M5; "heavy" = пропустить M3

report_path = run_sbs_evaluation(
    data=final_df,
    config=config,
    segment_name="OMTPL_Retail_Classical_FREQ",
    output_dir=str(REPORTS_DIR),
    size=RUN_SIZE,
)

print(report_path)

In [ ]:
from IPython.display import FileLink, display

report_file = Path(report_path)
print(f"Report exists: {report_file.exists()}")
print(f"Report size: {report_file.stat().st_size / 1024 / 1024:.2f} MB")
display(FileLink(str(report_file)))